# Downsampling Eaglei Data and Merging with County and ERA5 Data

The goal of this notebook is to:
- Downsample the eaglei data for each year to a 6-hour cadence, replacing the number of customers without power with the 6-hour mean
- Combine data for all years into a single file
- Export the combined data
- Merge additional county info
- Export the merged data

What I don't have working yet:
- Look up info from the ERA5 dataset about temperature, precipitation, snow depth, and wind speed

(I'm getting stuck on merging the weatherbench xarray with the eaglei data)

In [1]:
import pandas as pd
import numpy as np
import os

## Generate a list of all fips_codes

It turns out that some counties/fips_codes had no outage data for some years. We're going to interpret this as indicating that the county didn't have any power outages. Unfortunately, this means that for each year of the eaglei data, we'll need to identify the fips codes that lack data so we can manually add them to the to-be-filled dataframe later.

(Note that we're operating by year instead of all at once due to speed and memory issues), which even dask can't fix

In [ ]:
#Get a list of files that start with eaglei_outages_ in the directory ../Data/eaglei_data/
files = os.listdir('../Data/eaglei_data/')
files = [f for f in files if f.startswith('eaglei_outages_2')]

#Create an empty set
fips_codes = set()

#Create an empty data frame with the variables fips and year
fips_codes_df = pd.DataFrame(columns=['year', 'fips'])

i=0

for filename in files:
    #Open the file
    df = pd.read_csv('../Data/eaglei_data/' + filename)

    #Extract a sequence of four digits from filename
    year_local = int(filename.split('_')[2].split('.')[0])

    #Create a set of fips_code values for the loaded file
    fips_codes_local = set(df['fips_code'].unique())

    #Add the new set of fips_codes_local to the fips_codes set
    fips_codes = fips_codes.union(fips_codes_local)

    # Add a row to fips_codes_df with year=year_local and fips=fips_codes_local
    fips_codes_df.loc[i] = [year_local, fips_codes_local]
    i=i+1

fips_codes_missing_df = pd.DataFrame(columns=['year', 'fips_missing'])

i=0
for i in range(len(fips_codes_df)):
    fips_codes_local = fips_codes_df.loc[i]['fips']
    year = fips_codes_df.loc[i]['year']
    # Create a set of values that are in fips_codes but not in fips_codes_local
    missing_fips = fips_codes - fips_codes_local

    fips_codes_missing_df.loc[i] = [year, missing_fips]

    i=i+1

#In fips_codes_missing_df convert year to the index
#fips_codes_missing_df.set_index('year', inplace=True)

### Helper functions for downsampling

The eaglei data only includes entries when the customers_out value is nonzero.

In order to fill in all the missing values, we'll need to add additional (zero) entries to the eaglei data for the first and last possible timestamp of each year

The functions below are used to assist in this process

In [3]:
# Get the year from the filename as the four digits before .csv
def get_year_from_filename(filename):
    return int(filename.split('_')[2].split('.')[0])

# For each value of fips_code, if there isn't a value of run_start_time equal to start, add a row with run_start_time=start, fips_code, and customers_out=0
def add_missing_rows(df, start, end):
    # Create a date range from start to end with 15 minute intervals
    date_range = pd.date_range(start, end, freq='15min')

    # Create a DataFrame with all combinations of fips_code and the date range
    fips_codes = df['fips_code'].unique()
    all_combinations = pd.MultiIndex.from_product([fips_codes, date_range], names=['fips_code', 'run_start_time'])
    all_df = pd.DataFrame(index=all_combinations).reset_index()

    # Merge the original DataFrame with the new DataFrame to find missing rows
    merged_df = pd.merge(all_df, df, on=['fips_code', 'run_start_time'], how='left')

    # Fill missing values in customers_out with 0
    merged_df['customers_out'] = merged_df['customers_out'].fillna(0)

    # Set run_start_time as the index
    merged_df.set_index('run_start_time', inplace=True)

    return merged_df

# Add missing rows for each fips_code
#df_with_missing_rows = add_missing_rows(df_2021, start, end)


## Downsample the eaglei data

The data appear to only exist when the number of customers without power is nonzero. So we'll first need to upsample to a true 15-minute cadence and fill in missing values with 0.
(Note that this is a bit of an assumption that only non-zero values are being reported, rather than values being missing.)

Then we'll downsample to a 6-hour cadence by computing the mean number of customers out over the 6-hour window.

Note that this code chunk takes roughly 7 minutes to run.

In [40]:
#Get a list of files that start with eaglei_outages_ in the directory ../Data/eaglei_data/
#files = os.listdir('../Data/eaglei_data/')
#files = [f for f in files if f.startswith('eaglei_outages_2')]

files = ['eaglei_outages_2014.csv',
         'eaglei_outages_2015.csv', 
         'eaglei_outages_2016.csv',
         'eaglei_outages_2017.csv', 
         'eaglei_outages_2018.csv',
         'eaglei_outages_2019.csv',
         'eaglei_outages_2020.csv',
         'eaglei_outages_2021.csv',
         'eaglei_outages_2022.csv',
         'eaglei_outages_2023.csv']

#Create an empty data frame
outages = pd.DataFrame()

for filename in files:
    #Open the file
    df = pd.read_csv('../Data/eaglei_data/' + filename)

    #Drop the county and state variables to facilitate up/downsampling (these could be merged back in later if needed)
    df.drop(['county', 'state'], axis=1, inplace=True)

    #Convert the run_start_time variable into datetime
    df['run_start_time'] = pd.to_datetime(df['run_start_time'])
    
    # Identify the year from the filename
    year = get_year_from_filename(filename)

    # Create datetimes for the first and last timestamps of the year
    start = pd.to_datetime(f'{year}-01-01 00:00:00')
    end = pd.to_datetime(f'{year}-12-31 23:45:00')

    # Get the value of fips from year in fips_codes_df
    missing_fips = fips_codes_missing_df.loc[fips_codes_df['year'] == year, 'fips_missing'].values[0]

    # For each value in missing_fips, add a row to df with fips_code=value, customers_out=0, and run_start_time=start
    for fips_code in missing_fips:
        new_row = pd.DataFrame({'fips_code': [fips_code], 'customers_out': [0], 'run_start_time': [start]})
        df = pd.concat([df, new_row], ignore_index=True)

    # Add missing rows for each fips_code
    df = add_missing_rows(df, start, end)

    #In the fips_code variable, replace 0 with NA
    df['fips_code'] = df['fips_code'].replace(0, np.nan)

    #Forward fill the fips_code with the most recent value
    df['fips_code'] = df['fips_code'].ffill()

    #Group the data by fips_code. Then downsample the data to every 6 hours, replacing customers_out with the mean and then ungroup the data
    df = df.groupby('fips_code').resample('6h').mean().reset_index(level=0, drop=True)

    #Concatenate df with outages
    outages = pd.concat([outages, df])

#For exporting and future merging, move the datetime back to a "regular" variable and reset the index
outages['datetime'] = pd.to_datetime(outages.index)
outages.reset_index(drop=True, inplace=True)

### Bogus Code for Downsampling and Merging eaglei

The code below appears to leave the data with some non-filled-in zeroes...

In [ ]:
#Get a list of files that start with eaglei_outages_ in the directory ../Data/eaglei_data/
files = os.listdir('../Data/eaglei_data/')
files = [f for f in files if f.startswith('eaglei_outages_2')]

#Create an empty data frame
outages = pd.DataFrame()

for filename in files:
    #Open the file
    df = pd.read_csv('../Data/eaglei_data/' + filename)

    #Drop the county and state variables to facilitate up/downsampling (these could be merged back in later if needed)
    df.drop(['county', 'state'], axis=1, inplace=True)

    #Convert the run_start_time variable into datetime
    df['run_start_time'] = pd.to_datetime(df['run_start_time'])
    df.set_index('run_start_time', inplace=True)

    #Group the data by fips_code. Add new values of run_start_time every 15 minutes.
    # Forward fill values of customers_out with 0
    # Then ungroup the data
    df = df.groupby('fips_code').resample('15min').asfreq().fillna(0).reset_index(level=0, drop=True)

    #In the fips_code variable, replace 0 with NA
    df['fips_code'] = df['fips_code'].replace(0, np.nan)

    #Forward fill the fips_code with the most recent value
    df['fips_code'] = df['fips_code'].ffill()

    #Group the data by fips_code. Then downsample the data to every 6 hours, replacing customers_out with the mean and then ungroup the data
    df = df.groupby('fips_code').resample('6h').mean().reset_index(level=0, drop=True)


    #Concatenate df with outages
    outages = pd.concat([outages, df])

#For exporting and future merging, move the datetime back to a "regular" variable and reset the index
outages['datetime'] = pd.to_datetime(outages.index)
outages.reset_index(drop=True, inplace=True)


In [43]:
#Export outages to a parquet
outages.to_parquet('../Data/eaglei_data/eaglei_outages.parquet')

In [3]:
#Or, if the parquet file has already been saved, load the parquet
outages = pd.read_parquet('../Data/eaglei_data/eaglei_outages.parquet')

Next, we can merge the outages data with the additional county variables from the Counties_All file

In [44]:
#Add a YEAR variable from datetime
outages['YEAR'] = outages['datetime'].dt.year

#Load ../Data/Counties_All.csv
counties = pd.read_csv('../Data/County_level_Variables/Counties_All.csv')

# Merge outages and counties based on the YEAR variable and the fips_code/FIPS variables
outages_merged = outages.merge(counties, left_on=['YEAR', 'fips_code'], right_on=['YEAR', 'FIPS'])

In [45]:
#Export outages_merged to a parquet
outages_merged.to_parquet('../Data/Merged_Data/eaglei_outages_with_county_info.parquet')

In [ ]:
#Or, if the parquet file has already been saved, load the parquet
outages_merged = pd.read_parquet('../Data/Merged_Data/eaglei_outages_with_county_info.parquet')

## Checking for Missing Counties

It looks like some counties are missing data for some years. We'll check on this.

We'll:
- Make a dataframe (fips_codes_df) that lists the fips_code values present in each year's dataset and creates a set of all fips_code values from all years
- Make a dataframe (fips_codes_missing_df) that lists the fips_code values that don't have any data for each year
- Make a dataframe (missing_fips_df) that has one row per missing fips_code and indicates whether or not the code was missing in each year (1) or not (0) from 2014 to 2023

In [ ]:
#Get a list of files that start with eaglei_outages_ in the directory ../Data/eaglei_data/
files = os.listdir('../Data/eaglei_data/')
files = [f for f in files if f.startswith('eaglei_outages_2')]

#Create an empty set
fips_codes = set()

#Create an empty data frame with the variables fips and year
fips_codes_df = pd.DataFrame(columns=['year', 'fips'])

i=0

for filename in files:
    #Open the file
    df = pd.read_csv('../Data/eaglei_data/' + filename)

    #Extract a sequence of four digits from filename
    year_local = int(filename.split('_')[2].split('.')[0])

    #Create a set of fips_code values for the loaded file
    fips_codes_local = set(df['fips_code'].unique())

    #Add the new set of fips_codes_local to the fips_codes set
    fips_codes = fips_codes.union(fips_codes_local)

    # Add a row to fips_codes_df with year=year_local and fips=fips_codes_local
    fips_codes_df.loc[i] = [year_local, fips_codes_local]
    i=i+1

In [ ]:
fips_codes_missing_df = pd.DataFrame(columns=['year', 'fips_missing'])

i=0
for i in range(len(fips_codes_df)):
    fips_codes_local = fips_codes_df.loc[i]['fips']
    year = fips_codes_df.loc[i]['year']
    # Create a set of values that are in fips_codes but not in fips_codes_local
    missing_fips = fips_codes - fips_codes_local

    fips_codes_missing_df.loc[i] = [year, missing_fips]

    i=i+1

#In fips_codes_missing_df convert year to the index
fips_codes_missing_df.set_index('year', inplace=True)

In [ ]:
# Create a set that is the union of all sets in fips_codes_missing_df['fips_missing']
missing_fips = set()
for i in range(len(fips_codes_missing_df)):
    missing_fips = missing_fips.union(fips_codes_missing_df.loc[i]['fips_missing'])

# Create a pandas dataframe with columns fips, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, and 2023
missing_fips_df = pd.DataFrame(columns=['fips', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023'])

# For each item in missing_fips_years, create a list that consists of the key, a 1 for each year (in 2014-2023) that is in the values, and a 0 for each year that is not in the values
i=0
for fips in missing_fips:
    yearlist = [fips]
    # Create a list of 0s and 1s for each year from 2014 to 2023
    for year in range(10):
        currentyear = 2014+year
        tempset = fips_codes_missing_df.loc[currentyear]['fips_missing']
        if fips in tempset:
            yearlist.append(1)
        else:
            yearlist.append(0)
        # Add the fips code and the list of values to the dataframe
    missing_fips_df.loc[i] = yearlist
    i=i+1

# Sort missing_fips_df by fips
missing_fips_df.sort_values(by='fips', inplace=True)

In [100]:
# Export missing_fips_df to a csv
missing_fips_df.to_csv('../Data/Missing_FIPS.csv', index=False)

In [ ]:
# We can also create a dictionary that uses the fips_code as a key and its missing years as values:
missing_fips_years = {}
for fips in missing_fips:
    missing_fips_years[fips] = []
    for i in range(len(fips_codes_missing_df)):
        if fips in fips_codes_missing_df.loc[i]['fips_missing']:
            missing_fips_years[fips].append(fips_codes_missing_df.loc[i]['year'])
